# Temperature Imputation Workflow Skeleton

This notebook provides a reusable framework for implementing and comparing imputation strategies.

Primary objective in this project:
- Build leakage-safe imputers on train data only
- Compare baseline and advanced imputers
- Support temperature-specific hierarchical fallback logic
- Export a reusable preprocessing + modeling pipeline

In [ ]:
# 1) Import Libraries and Configure Notebook
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Imports ready.")

In [ ]:
# 2) Load Dataset and Define Feature Types
DATA_DIR = Path("/home/adhil/github/EMULaToR/data/processed/baselines/CataPro/kcat")
TRAIN_PATH = DATA_DIR / "train.csv"  # or train.parquet
VAL_PATH = DATA_DIR / "val.csv"  # or val.parquet
TEST_PATH = DATA_DIR / "test.csv"  # or test.parquet

TARGET_COL = "log10_value"  # change to your training target if needed

# Optional schema harmonization map for this project
RENAME_MAP = {
    "sequence": "seq",
    "value": "kcat",
    "uniprot_id": "ProtID",
}

# Feature groups: adjust to your schema
ID_COLS = ["substrate_id", "ProtID"]
NUMERIC_COLS = ["kcat"]  # add temperature-derived numeric cols after creation
CATEGORICAL_COLS = ["unit", "source"]


def load_any_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file format: {path}")


def standardize_schema(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.rename(columns={k: v for k, v in RENAME_MAP.items() if k in out.columns})
    return out


train_df_raw = standardize_schema(load_any_table(TRAIN_PATH))
val_df_raw = standardize_schema(load_any_table(VAL_PATH))
test_df_raw = standardize_schema(load_any_table(TEST_PATH))

print("Train shape:", train_df_raw.shape)
print("Val shape:", val_df_raw.shape)
print("Test shape:", test_df_raw.shape)
train_df_raw.head(2)

In [ ]:
# 3) Profile Missingness


def missingness_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2),
            "dtype": df.dtypes.astype(str),
        }
    ).sort_values("missing_pct", ascending=False)
    return rep


train_missing = missingness_report(train_df_raw)
val_missing = missingness_report(val_df_raw)
test_missing = missingness_report(test_df_raw)

THRESHOLD_PCT = 30.0
high_missing_cols = train_missing[
    train_missing["missing_pct"] > THRESHOLD_PCT
].index.tolist()

print("Columns above threshold in train:", high_missing_cols)
display(train_missing.head(20))

plt.figure(figsize=(10, 4))
train_missing["missing_pct"].head(30).plot(kind="bar")
plt.title("Top 30 Missingness Percentages in Train")
plt.ylabel("Missing %")
plt.tight_layout()
plt.show()

In [ ]:
# 4) Train/Validation Split (for internal model selection if needed)
# If you already have external val.csv, keep it as-is and do not refit imputers on val/test.

WORKING_TARGET = TARGET_COL

# Keep a holdout from train only for local comparison among imputation strategies
train_local, valid_local = train_test_split(
    train_df_raw,
    test_size=0.2,
    random_state=SEED,
)

print("train_local:", train_local.shape, "valid_local:", valid_local.shape)

# Sanity check target presence
assert WORKING_TARGET in train_local.columns, f"Missing target column: {WORKING_TARGET}"

In [ ]:
# 5) Baseline Imputation (Mean/Median/Mode)
# This section demonstrates SimpleImputer per data type.

candidate_numeric = [
    c
    for c in train_local.columns
    if pd.api.types.is_numeric_dtype(train_local[c]) and c != WORKING_TARGET
]
candidate_categorical = [
    c for c in train_local.columns if c not in candidate_numeric + [WORKING_TARGET]
]

print("Numeric columns:", len(candidate_numeric))
print("Categorical columns:", len(candidate_categorical))

simple_num = SimpleImputer(strategy="median")
simple_cat = SimpleImputer(strategy="most_frequent")

X_train_local = train_local.drop(columns=[WORKING_TARGET])
y_train_local = train_local[WORKING_TARGET]
X_valid_local = valid_local.drop(columns=[WORKING_TARGET])
y_valid_local = valid_local[WORKING_TARGET]

X_train_simple = X_train_local.copy()
X_valid_simple = X_valid_local.copy()

if candidate_numeric:
    X_train_simple[candidate_numeric] = simple_num.fit_transform(
        X_train_local[candidate_numeric]
    )
    X_valid_simple[candidate_numeric] = simple_num.transform(
        X_valid_local[candidate_numeric]
    )

if candidate_categorical:
    X_train_simple[candidate_categorical] = simple_cat.fit_transform(
        X_train_local[candidate_categorical]
    )
    X_valid_simple[candidate_categorical] = simple_cat.transform(
        X_valid_local[candidate_categorical]
    )

print("Missing after SimpleImputer (train):", int(X_train_simple.isna().sum().sum()))
print("Missing after SimpleImputer (valid):", int(X_valid_simple.isna().sum().sum()))

In [ ]:
# 6) Advanced Imputation (KNN / Iterative)
# KNN/Iterative require numeric-only matrices. We compare them on numeric columns.

advanced_results = []

if candidate_numeric:
    Xtr_num = X_train_local[candidate_numeric].copy()
    Xva_num = X_valid_local[candidate_numeric].copy()

    # KNNImputer
    knn_imp = KNNImputer(n_neighbors=5, weights="distance")
    Xtr_knn = pd.DataFrame(
        knn_imp.fit_transform(Xtr_num), columns=candidate_numeric, index=Xtr_num.index
    )
    Xva_knn = pd.DataFrame(
        knn_imp.transform(Xva_num), columns=candidate_numeric, index=Xva_num.index
    )

    # IterativeImputer
    iter_imp = IterativeImputer(
        estimator=BayesianRidge(),
        random_state=SEED,
        max_iter=20,
        sample_posterior=False,
    )
    Xtr_iter = pd.DataFrame(
        iter_imp.fit_transform(Xtr_num), columns=candidate_numeric, index=Xtr_num.index
    )
    Xva_iter = pd.DataFrame(
        iter_imp.transform(Xva_num), columns=candidate_numeric, index=Xva_num.index
    )

    # Quick proxy comparison: train a small model on imputed numeric-only data
    rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)

    rf.fit(Xtr_knn, y_train_local)
    pred_knn = rf.predict(Xva_knn)
    advanced_results.append(
        {
            "imputer": "KNNImputer",
            "rmse": mean_squared_error(y_valid_local, pred_knn, squared=False),
            "mae": mean_absolute_error(y_valid_local, pred_knn),
            "r2": r2_score(y_valid_local, pred_knn),
        }
    )

    rf.fit(Xtr_iter, y_train_local)
    pred_iter = rf.predict(Xva_iter)
    advanced_results.append(
        {
            "imputer": "IterativeImputer",
            "rmse": mean_squared_error(y_valid_local, pred_iter, squared=False),
            "mae": mean_absolute_error(y_valid_local, pred_iter),
            "r2": r2_score(y_valid_local, pred_iter),
        }
    )

pd.DataFrame(advanced_results)

In [ ]:
# 7) ColumnTransformer + Pipeline Setup
# This is the reusable production-style setup.

numeric_features = candidate_numeric
categorical_features = candidate_categorical

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

model = RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1)

full_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ]
)

print("Pipeline ready.")

In [ ]:
# 8) Model Training on Imputed Data
full_pipeline.fit(X_train_local, y_train_local)

pred_valid = full_pipeline.predict(X_valid_local)

metrics = {
    "rmse": mean_squared_error(y_valid_local, pred_valid, squared=False),
    "mae": mean_absolute_error(y_valid_local, pred_valid),
    "r2": r2_score(y_valid_local, pred_valid),
}

print("Validation metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.6f}")

In [ ]:
# 9) Imputation Quality and Model Evaluation
# Compare simple baseline vs advanced numeric imputers and full pipeline metrics.

summary_rows = []

if advanced_results:
    summary_rows.extend(advanced_results)

summary_rows.append(
    {
        "imputer": "SimpleImputer + ColumnTransformer + RF",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "r2": metrics["r2"],
    }
)

results_df = pd.DataFrame(summary_rows).sort_values("rmse")
display(results_df)

plt.figure(figsize=(7, 4))
sns.barplot(data=results_df, x="imputer", y="rmse")
plt.xticks(rotation=20, ha="right")
plt.title("RMSE by Imputation Strategy")
plt.tight_layout()
plt.show()

In [ ]:
# Project-Specific: Hierarchical Temperature Imputation Skeleton
# Fit mapping tables on train only, then apply to val/test without leakage.

TEMP_COL = "Temp"  # set this to your source temperature column in Celsius
PROT_COL = "ProtID"
SUB_COL = "substrate_id"
EC_COL = "EC"  # optional; if absent, EC-level fallback is skipped


def to_temperature_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if TEMP_COL not in out.columns:
        out[TEMP_COL] = np.nan
    out[TEMP_COL] = pd.to_numeric(out[TEMP_COL], errors="coerce")
    return out


def fit_temp_imputer(train_df: pd.DataFrame) -> dict:
    tdf = to_temperature_frame(train_df)

    global_median = float(tdf[TEMP_COL].median())

    by_pair = None
    if PROT_COL in tdf.columns and SUB_COL in tdf.columns:
        by_pair = tdf.groupby([PROT_COL, SUB_COL], dropna=True)[TEMP_COL].median()

    by_prot = None
    if PROT_COL in tdf.columns:
        by_prot = tdf.groupby(PROT_COL, dropna=True)[TEMP_COL].median()

    by_ec = None
    if EC_COL in tdf.columns:
        by_ec = tdf.groupby(EC_COL, dropna=True)[TEMP_COL].median()

    return {
        "global_median": global_median,
        "by_pair": by_pair,
        "by_prot": by_prot,
        "by_ec": by_ec,
    }


def apply_temp_imputer(
    df: pd.DataFrame, stats: dict, clip_range=(0.0, 100.0)
) -> pd.DataFrame:
    out = to_temperature_frame(df)

    def _fallback(row):
        if pd.notna(row[TEMP_COL]):
            return row[TEMP_COL]

        if stats["by_pair"] is not None:
            key = (row.get(PROT_COL), row.get(SUB_COL))
            if key in stats["by_pair"].index and pd.notna(stats["by_pair"].loc[key]):
                return float(stats["by_pair"].loc[key])

        if stats["by_prot"] is not None:
            key = row.get(PROT_COL)
            if key in stats["by_prot"].index and pd.notna(stats["by_prot"].loc[key]):
                return float(stats["by_prot"].loc[key])

        if stats["by_ec"] is not None:
            key = row.get(EC_COL)
            if key in stats["by_ec"].index and pd.notna(stats["by_ec"].loc[key]):
                return float(stats["by_ec"].loc[key])

        return float(stats["global_median"])

    out[TEMP_COL] = out.apply(_fallback, axis=1)
    out[TEMP_COL] = out[TEMP_COL].clip(clip_range[0], clip_range[1])

    # ProKcat-compatible derived columns
    out["Temp_K"] = out[TEMP_COL] + 273.15
    out["Inv_Temp"] = 1.0 / out["Temp_K"]

    # Fixed normalization bounds used by ProKcat data files
    Tmin, Tmax = 273.15, 373.15
    inv_min, inv_max = 1.0 / Tmax, 1.0 / Tmin
    out["Temp_K_norm"] = (out["Temp_K"] - Tmin) / (Tmax - Tmin)
    out["Inv_Temp_norm"] = (out["Inv_Temp"] - inv_min) / (inv_max - inv_min)

    return out


temp_stats = fit_temp_imputer(train_df_raw)
train_df = apply_temp_imputer(train_df_raw, temp_stats)
val_df = apply_temp_imputer(val_df_raw, temp_stats)
test_df = apply_temp_imputer(test_df_raw, temp_stats)

print("Missing Temp after imputation:")
print(
    {
        "train": int(train_df[TEMP_COL].isna().sum()),
        "val": int(val_df[TEMP_COL].isna().sum()),
        "test": int(test_df[TEMP_COL].isna().sum()),
    }
)

In [ ]:
# 10) Persist Pipeline and Reuse on New Data
import joblib

ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_path = ARTIFACT_DIR / "imputation_model_pipeline.joblib"
joblib.dump(full_pipeline, pipeline_path)
print("Saved:", pipeline_path)

loaded_pipeline = joblib.load(pipeline_path)
_ = loaded_pipeline.predict(X_valid_local.head(10))
print("Reloaded pipeline works.")

In [ ]:
# Export ProKcat-Ready Splits
OUTPUT_DIR = Path("/home/adhil/github/ProKcat/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep/add whichever columns your ProKcat preparation needs.
required_cols = [
    "smiles",
    "seq",
    "kcat",
    "ProtID",
    "Temp",
    "Temp_K",
    "Inv_Temp",
    "Temp_K_norm",
    "Inv_Temp_norm",
]


def ensure_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = np.nan
    return out


train_out = ensure_columns(train_df, required_cols)
val_out = ensure_columns(val_df, required_cols)
test_out = ensure_columns(test_df, required_cols)

train_out.to_csv(OUTPUT_DIR / "train_data.csv", index=False)
val_out.to_csv(OUTPUT_DIR / "val_data.csv", index=False)
test_out.to_csv(OUTPUT_DIR / "test_data.csv", index=False)

print("Saved processed splits to", OUTPUT_DIR)

## TODO: Project-Specific Tweaks

- Confirm the true source temperature column name and units.
- Decide fallback group keys and order for temperature imputation.
- Add clipping policy for unrealistic assay temperatures.
- Validate distributions of Temp and Temp_K_norm after imputation.
- Run sensitivity checks for fallback choices (for example global median vs fixed 25C/30C/37C).
- Confirm exported columns exactly match downstream ProKcat feature-generation scripts.